# Check my setup

Run the cells from top to bottom. Each test prints PASS or FAIL and says what a failure means.

* **Test 1** checks that `hsiplot` is installed and the analysis gives right answers on made-up data. No SDK and no data needed.
* **Test 2** checks the Living Optics SDK on your computer. It reports what the SDK can and cannot do.
* **Test 3** (optional) runs the analysis on your own table of spectra.
* **Test 4** (optional) decodes one `.lo` file. It needs an SDK that includes the spectral decoder.

In [ ]:
import hsiplot

print("hsiplot", hsiplot.__version__)
RESULTS = {}

## Test 1: the analysis works and gives the right answer on made-up data

In [ ]:
from hsiplot import PreprocessConfig, group_difference_test, make_synthetic_table, preprocess_table, run_classification

# A: a table with a real built-in effect must be recognised on plots the model has not seen
real = make_synthetic_table(n_plots=12, n_control=2, n_spectra=60, n_bands=48, effect=0.5, plot_sd=0.05, seed=1)
real = preprocess_table(real, PreprocessConfig.preset("clean")).subsample(60, seed=0)
r = run_classification(real, n_estimators=30)
print("known effect: plot AUC", round(r.statistic, 2), " permutation p", round(r.p_value, 3))
ok_a = r.statistic >= 0.9

# B: a table with NO effect must NOT look significant
none = make_synthetic_table(n_plots=12, n_control=2, n_spectra=60, n_bands=48, effect=0.0, plot_sd=0.3, seed=7)
none = preprocess_table(none).subsample(40)
r0 = run_classification(none, n_estimators=30)
print("no effect:    plot AUC", round(r0.statistic, 2), " permutation p", round(r0.p_value, 3))
ok_b = r0.p_value > 0.05

# C: PERMANOVA agrees with A
g = group_difference_test(real)
print("PERMANOVA on the known effect: p", round(g.p_value, 3))
ok_c = g.p_value < 0.05

RESULTS["Test 1"] = ok_a and ok_b and ok_c
print("PASS" if RESULTS["Test 1"] else "FAIL: unexpected result; reinstall with pip install -e .[dev] and run pytest")

## Test 2: the Living Optics SDK on this computer

In [ ]:
import importlib


def has(mod, name=None):
    try:
        m = importlib.import_module(mod)
        return (name is None) or hasattr(m, name)
    except Exception:
        return False

sdk = {
    "open a .lo file (lo.sdk.api.acquisition.io.open)": has("lo.sdk.api.acquisition.io.open", "open"),
    "RGB converter (LORAWtoRGB8)": has("lo.sdk.api.acquisition.data.formats", "LORAWtoRGB8"),
    "spectral decoder (SpectralDecoder)": has("lo.sdk.api.acquisition.data.decode", "SpectralDecoder"),
}
for k, v in sdk.items():
    print(("found   " if v else "MISSING ") + k)
CAN_DECODE = all(sdk.values())
print()
if CAN_DECODE:
    print("This SDK can decode spectra. Run Test 4.")
else:
    print("This SDK cannot decode spectra, so .lo files cannot be turned into spectra here.")
    print("That is a limit of the SDK edition installed, not of hsiplot. Everything after extraction still works (Test 1 and 3).")

## Test 3 (optional): your own table of spectra

Set `TABLE` to a CSV or XLSX with one row per spectrum, a treatment column and band columns. The settings shown suit a table with a `Herbicide` column and 4,384 rows per plot; adjust them to your table. Leave `TABLE = None` to skip.

In [ ]:
TABLE = None   # for example: r"C:/path/to/spectra.xlsx"  (or a .csv)

if TABLE:
    from hsiplot import load_table, plot_qc
    t = load_table(TABLE, group_col="Herbicide", control_label="Control", rows_per_plot=4384,
                   wavelength_range=(440, 900), quantity="radiance")
    print(len(t.plot_table()), "plots,", len(t.bands), "bands,", len(t.df), "spectra")
    display(plot_qc(t))          # look for plots marked CHECK
    t = preprocess_table(t, PreprocessConfig.preset("clean")).subsample(300, seed=0)
    res = run_classification(t, n_estimators=100)
    print(res.summary())
    display(res.plot_scores)
    print(group_difference_test(t).summary())
    RESULTS["Test 3"] = True
else:
    print("skipped")

## Test 4 (optional): decode one `.lo` file

Needs an SDK that includes the spectral decoder and the factory calibration folder from Living Optics. Leave both as `None` to skip.

In [ ]:
LO_FILE = None            # for example: r"C:/data/plot01.lo"
CALIBRATION_FOLDER = None # the factory calibration folder for your camera

if LO_FILE and CALIBRATION_FOLDER:
    from hsiplot.extract import check_file
    rep = check_file(LO_FILE, CALIBRATION_FOLDER)
    for step, status in rep["steps"]:
        print(status[:60].ljust(60), step)
    print({k: v for k, v in rep.items() if k not in ("steps",)})
    RESULTS["Test 4"] = all(s == "ok" for _, s in rep["steps"])
else:
    print("skipped")

## Summary

In [ ]:
for k, v in RESULTS.items():
    print(k, "PASS" if v else "FAIL")
print("SDK can decode:", CAN_DECODE)